In [4]:
from pathlib import Path
import pandas as pd


base_data_path = Path("/mnt/projects/lihao_project/")

file_name = "cleaned_Crimes_-_2001_to_Present_20250114.csv"

In [5]:
df = pd.read_csv(base_data_path / file_name)


In [6]:
df.Date = pd.to_datetime(df.Date)
df['Updated On'] = pd.to_datetime(df['Updated On'])

In [9]:
import geopandas as gpd
import numpy as np


community_areas = gpd.read_file("./data/chicago_community_areas/geo_export_a6df0e5e-f6bd-41de-bed3-2d329795f76a.shp")
#  State Plane Illinois East NAD 1983 projection to match the X,Y coordinates on crime data.
community_areas = community_areas.to_crs('EPSG:3435') 
# community_areas['beat_num'] = community_areas['beat_num'].astype(int)

# Calculate the centroid of community_areas
community_areas['centroid_x'] = community_areas.centroid.x
community_areas['centroid_y'] = community_areas.centroid.y

community_areas.rename({'area_num_1': 'area_id'}, axis=1, inplace=True)
community_areas['area_id'] = community_areas['area_id'].astype(int)


unique_community_areas = community_areas['area_id'].unique()

np.setdiff1d( df['Community Area'].astype('int').unique(), unique_community_areas)

array([0])

In [10]:
# Bin the data to a time series
def bin_data_by_time(df, time_unit, interval=1):
    
    df = df.copy()
    if time_unit == 'day':
        df['time_bin'] = df['Date'].dt.to_period(f'{interval}D').dt.start_time
    elif time_unit == 'week':
        df['time_bin'] = df['Date'].dt.to_period(f'{interval}W').dt.start_time
    elif time_unit == 'month':
        df['time_bin'] = df['Date'].dt.to_period(f'{interval}M').dt.start_time
    elif time_unit == 'year':
        df['time_bin'] = df['Date'].dt.to_period(f'{interval}Y').dt.start_time
    else:
        raise ValueError("Invalid time unit. Choose from 'day', 'week', 'month', 'year'.")
    

    df['year'] = df['time_bin'].dt.year
    if time_unit in ['month', 'day', 'week']:
        df['month'] = df['time_bin'].dt.month
    if time_unit == 'day':
        df['day'] = df['time_bin'].dt.day
    
    return df

def add_time_columns(df, time_unit):
    df['year'] = df['time_bin'].dt.year
    if time_unit in ['month', 'day', 'week']:
        df['month'] = df['time_bin'].dt.month
    if time_unit == 'day':
        df['day'] = df['time_bin'].dt.day
    


In [11]:

time_unit = 'month' # Can be (day, week, month, year)
interval = 1

df_binned = bin_data_by_time(df, time_unit, interval)

crime_by_community_areas_time = df_binned.groupby(['time_bin', 'Community Area']).size().reset_index(name='crime_count')

display(crime_by_community_areas_time)

,time_bin,Community Area,crime_count
0,2001-01-01,1.0,701
1,2001-01-01,2.0,457
2,2001-01-01,3.0,687
3,2001-01-01,4.0,282
4,2001-01-01,5.0,283
...,...,...,...
22298,2025-01-01,73.0,27
22299,2025-01-01,74.0,5
22300,2025-01-01,75.0,19
22301,2025-01-01,76.0,17


In [16]:
crime_data_time = community_areas.merge(crime_by_community_areas_time, left_on='area_id', right_on='Community Area', how='left').fillna(0)[['area_id', 'centroid_x', 'centroid_y', 'time_bin', 'crime_count']]

add_time_columns(crime_data_time, time_unit)

crime_data_time = crime_data_time.set_index('area_id')
crime_data_time = crime_data_time[crime_data_time['year'] < 2025]
crime_data_time = crime_data_time.sort_values(by = ['time_bin', 'area_id'])
crime_data_time.to_csv(f"crime_data_by_community_areas_{interval}_{time_unit}.csv")

In [17]:
crime_data_time

,centroid_x,centroid_y,time_bin,crime_count,year,month
area_id,,,,,,
1,1.164480e+06,1.946795e+06,2001-01-01,701,2001,1
2,1.157750e+06,1.943809e+06,2001-01-01,457,2001,1
3,1.168490e+06,1.930860e+06,2001-01-01,687,2001,1
4,1.159860e+06,1.934204e+06,2001-01-01,282,2001,1
5,1.160937e+06,1.924235e+06,2001-01-01,283,2001,1
...,...,...,...,...,...,...
73,1.171110e+06,1.840389e+06,2024-12-01,203,2024,12
74,1.153615e+06,1.832015e+06,2024-12-01,39,2024,12
75,1.165684e+06,1.830229e+06,2024-12-01,153,2024,12
